The goal of this notebook is to train some adversarial patches on the CIFAR10 data set and visualize what these patches look like, as well as explore the effectiveness of patches once we alter certain features.

This notebook is largely modelled after: https://github.com/AIPI-590-XAI/Duke-AI-XAI/blob/main/adversarial-ai-example-notebooks/adversarial_attacks_patches.ipynb

I am writing my own version of tihs notebook to build a stronger understanding of this work.

In [8]:
# Install necessary packages if they do not exist.
# %pip install --upgrade numpy
# %pip install numpy matplotlib torch torchvision
# %pip install --upgrade pip

In [9]:
import os
import numpy as np
import math

import matplotlib as plt

import torch
from torch.utils.data import DataLoader

import torchvision
from torchvision.models import resnet50, ResNet50_Weights
from torchvision import datasets, transforms

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

device = torch.device("cpu") if not torch.cuda.is_available() else torch.device("cuda:0")


GPU available: False


There exist many kinds of models that torchvision offers. In terms of image classification, ResNet50 is mainstream. Choosing default pre-trained weights for ResNet50 forces the most up-to-date version.

In [10]:
pretrained_model = torchvision.models.resnet50(weights = ResNet50_Weights.DEFAULT)
pretrained_model.to(device) # moves model to device

for p in pretrained_model.parameters():
    p.requires_grad = False

We need to normalize input. Color channel intensity is measured by z scores as opposed to their actual values. 

Numbers for mean and standard deviation can be found here: https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet50.html

In [11]:
# Mean and Std from ImageNet
NORM_MEAN = np.array([0.485, 0.456, 0.406])
NORM_STD = np.array([0.229, 0.224, 0.225])


plain_transforms = transforms.Compose([
    transforms.ToTensor(), # Maps RGB to (0,1) as opposed to (0,255)
    transforms.Normalize(mean=NORM_MEAN,
                         std=NORM_STD) # 
])





We first want to evaluate how our model performs on unperturbed images. TinyImageNet contains 1000 separate labels, each with 5 images which are drawn from ImageNet. ResNet50's pretrained weights are trained on IMAGENET1K_V2 (as of October 13th, 2026)

In [12]:
imagenet_path = "./data/TinyImageNet"

dataset = datasets.ImageFolder(
    root = imagenet_path,
    transform = plain_transforms
    )

data_loader = DataLoader(
    dataset,
    batch_size = 32,
    shuffle = True,
    drop_last = False
)

print("Images:", len(dataset))
print("Classes:", len(dataset.classes))

images, labels = next(iter(data_loader))
print("Batch shape:", images.shape)

Images: 5000
Classes: 1000
Batch shape: torch.Size([32, 3, 224, 224])


Now let's evaluate the model:

In [ ]:
def evaluate_model(model, data_loader, device):
    model.eval()
    correct_top1 = 0
    correct_top5 = 0
    total = 0


    with torch.no_grad():
        for images, labels in data_loader:
            # inputs a image batch of 32. Returns [batchsize, 1000].
            # Each entry in 1000 represents the probabiltiity that our model attributes.
            scores = model(images) 

            # index of maximum value along horizontal axis, which is likelihoods
            predictions = scores.argmax(dim = 1) 
            # item() returns the number into a python integer as opposed to a tensor num
            correct_top1 += (predictions == labels).sum().item() 

            # finds top 5 scoring indices
            top5_predictions = scores.topk(5, dim=1).indices 
            correct_top5 += (
                top5_predictions == labels.unsqueeze(dim = 1)
            ).any(dim = 1).sum().item()

            total += labels.size(0)

    return correct_top1 / total, correct_top5 / total

top1, top5 = evaluate_model(pretrained_model, data_loader, device)

print(f"Top-1 Accuracy {top1:.2%}")
print(f"Top-5 Accuracy {top5:.2%}")

